In [ ]:
import numpy as np

np.random.seed(42)
n_samples_per_class = 100
class1 = np.random.multivariate_normal([2, 2], [[1, 0.5], [0.5, 1]], n_samples_per_class)
class2 = np.random.multivariate_normal([-2, -2], [[1, 0.5], [0.5, 1]], n_samples_per_class)
x = np.vstack((class1, class2))
y = np.hstack((np.zeros(n_samples_per_class), np.ones(n_samples_per_class)))

input_size = 2
hidden_size = 10
output_size = 1
w1 = np.random.randn(input_size,hidden_size) * 0.01
w2 = np.random.randn(hidden_size,output_size) * 0.01
b1 = np.random.randn(1,hidden_size) * 0.01
b2 = np.random.randn(1,output_size) * 0.01
N = x.shape[0]
def sigmoid(x):
    return 1 / (1 + np.exp(-x))
learning_rate = 0.01
epochs = 1000

for epoch in range(epochs):
    # ---------- Forward ----------
    Z1 = np.dot(x, w1) + b1
    A1 = np.tanh(Z1)
    Z2 = np.dot(A1, w2) + b2
    A2 = sigmoid(Z2)

    # ---------- Loss ----------
    loss = - (1/N) * np.sum(y.reshape(-1,1) * np.log(A2 + 1e-8) + (1 - y.reshape(-1,1)) * np.log(1 - A2 + 1e-8))

    # ---------- Backward ----------
    dZ2 = A2 - y.reshape(-1,1)
    dw2 = (1/N) * np.dot(A1.T, dZ2)
    db2 = (1/N) * np.sum(dZ2, axis=0, keepdims=True)

    dA1 = np.dot(dZ2, w2.T)
    dZ1 = dA1 * (1 - np.tanh(Z1)**2)
    dw1 = (1/N) * np.dot(x.T, dZ1)
    db1 = (1/N) * np.sum(dZ1, axis=0, keepdims=True)

    # ---------- Update ----------
    w1 -= learning_rate*dw1
    b1 -= learning_rate*db1
    w2 -= learning_rate*dw2
    b2 -= learning_rate*db2

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

predictions = (A2>0.5).astype(int).flatten()
accuracy = np.mean(predictions == y)
print(f"Accuracy: {accuracy * 100:.2f}%")

In [ ]:
import numpy as np


class NeuralNetwork:
    def __init__(self, input_size=2, hidden_size=10, output_size=1, learning_rate=0.1):
        self.learning_rate = learning_rate

        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))

        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros((1, output_size))

        self.loss_history = []

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def relu(self, z):
        return np.maximum(0, z)

    def relu_derivative(self, z):
        return (z > 0).astype(float)

    def forward(self, X):
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = self.relu(self.Z1)

        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.A2 = self.sigmoid(self.Z2)

        return self.A2

    def compute_loss(self, y, predictions):
        m = len(y)
        y = y.reshape(-1, 1)
        loss = -np.mean(y * np.log(predictions + 1e-8)+ (1 - y) * np.log(1 - predictions + 1e-8))
        return loss

    def backward(self, X, y):
        m = X.shape[0]
        y = y.reshape(-1, 1)

        dZ2 = self.A2 - y
        dW2 = (1 / m) * np.dot(self.A1.T, dZ2)
        db2 = (1 / m) * np.sum(dZ2, axis=0, keepdims=True)

        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * self.relu_derivative(self.Z1)

        dW1 = (1 / m) * np.dot(X.T, dZ1)
        db1 = (1 / m) * np.sum(dZ1, axis=0, keepdims=True)

        self.W1 -= self.learning_rate * dW1
        self.b1 -= self.learning_rate * db1
        self.W2 -= self.learning_rate * dW2
        self.b2 -= self.learning_rate * db2

    def train(self, X, y, epochs=1000):
        for epoch in range(epochs):
            predictions = self.forward(X)

            loss = self.compute_loss(y, predictions)
            self.loss_history.append(loss)

            self.backward(X, y)

            if epoch % 100 == 0:
                print(f"Epoch {epoch}, Loss: {loss:.4f}")

    def predict(self, X):
        return (self.forward(X) > 0.5).astype(int)